# ESP-LLM — Train ESP32-S3 model on Kaggle GPU (both T4s, DeepSpeed ZeRO-2)

Clones this repo and trains the **ESP32-S3 (~51 M-param, 12-layer MoE)** model on **both** Kaggle T4s with one command:
`deepspeed --num_gpus=2 main.py --target=esp32s3 --train --deepspeed`

> **Setup (2 clicks):** right panel `Settings > Accelerator > GPU T4 x2`, turn **Internet ON** (needed for `git clone` + `pip install`), then `Run all`.

## ⏱️ How long on Kaggle?

| GPU choice | Expected wall time (S3: 25k iters max, early stopping usually ends it at 12k–20k) |
|---|---|
| **T4 x2 + DeepSpeed ZeRO-2 (this notebook)** | **~5–9 h** typical (early-stopped); ~12–16 h if it ran all 25k iters |
| Single P100 (`python main.py --target=esp32s3 --train`) | ~7–13 h typical; ~11–16 h full |
| Single T4, no DeepSpeed | ~8–14 h typical; ~12–18 h full |

Why 2 GPUs ≈ 1.6–1.8× faster (not 2×): each rank trains its own batch of 32×192 tokens, so every step processes **12,288 tokens** (2× data) with ~10–20% NCCL sync overhead. ZeRO stage 2 shards optimizer states + gradients (params stay replicated, so the `.pt` save path is unchanged). `main.py` is FP32-only, so no mixed-precision boost — the win is pure data-parallel throughput.

## ⚠️ One-session fit (important)

- Kaggle free quota: **~30 GPU-h/week**, max **~9 h per session**. A typical early-stopped 2-GPU run (~5–9 h) now **fits in one session**; a worst-case full 25k-iter run does not.
- `main.py` always trains from scratch (`start_iter=0`, no resume), so a killed session restarts from zero. What survives is whatever you persist via **Save Version → Output** (see last cell) — the `.best` checkpoint from even a partial run is often already usable.
- Tip: prefer a batch run (**Save Version > Save & Run All**) over a long interactive session, so the 60-min idle timeout doesn't eat quota. Monitor quota at profile avatar > Settings > Quotas.

In [ ]:
# 1 — Verify BOTH T4 GPUs are visible (need 2 for the deepspeed run)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())
assert torch.cuda.device_count() == 2, 'Pick GPU T4 x2 in Settings > Accelerator, then re-run'
print(torch.cuda.get_device_name(0), '/', torch.cuda.get_device_name(1))

In [ ]:
# 2 — Clone the repo (re-runnable). Requires Internet=ON in Kaggle Settings.
!rm -rf /kaggle/working/espllm
!git clone https://github.com/ahmedbarakat207/espllm.git /kaggle/working/espllm
!ls -lh /kaggle/working/espllm | head -30
!du -h /kaggle/working/espllm/dataset.txt /kaggle/working/espllm/bpe-vocab.json /kaggle/working/espllm/bpe-merges.txt

In [ ]:
# 3 — Install deps (Kaggle image already has torch+CUDA; only add what's missing)
%cd /kaggle/working/espllm
!pip install -q tokenizers torchao deepspeed
!python -c "import deepspeed; print('deepspeed', deepspeed.__version__)"
!ls -lh ds_config_zero2.json

In [ ]:
# 4 — TRAIN on both GPUs (this is the only command you need)
# DeepSpeed ZeRO-2 data-parallel: each GPU takes batch 32×192, global 64×192
# = 12,288 tokens/step. Single-GPU `main.py` is untouched — this flag path
# forces standard AdamW (4-bit torchao AdamW is single-GPU only) and shards
# optimizer states + gradients across the 2 T4s (see ds_config_zero2.json).
%cd /kaggle/working/espllm
!deepspeed --num_gpus=2 main.py --target=esp32s3 --train --deepspeed --deepspeed_config=ds_config_zero2.json

# Fallback: single GPU (uses 2nd T4 for nothing, ~1.7× slower):
# !python main.py --target=esp32s3 --train

## After training — persist outputs

The run writes to `/kaggle/working/espllm/model/`:

- `model_esp32s3.pt` — full-precision weights
- `model_esp32s3.pt.best` — best early-stopped weights (lowest val loss)
- `model_esp32s3.pt.quantized` — post-training quantized artifact (what `convert_model_to_c.py` / `flash.py` consumes)

Kaggle wipes `/kaggle/working` when the session ends unless you **Save Version**. Use Quick Save to checkpoint without re-running, or copy the files below so they land in the version Output.

In [ ]:
# 5 — Verify + stage outputs so Save Version keeps them
%cd /kaggle/working/espllm
!ls -lh model/model_esp32s3* 2>/dev/null || echo 'no S3 checkpoint yet (training did not finish)'
# Stage a copy at /kaggle/working/output for the version snapshot:
!mkdir -p /kaggle/working/output && cp -f model/model_esp32s3.pt* /kaggle/working/output/ 2>/dev/null; ls -lh /kaggle/working/output/ 2>/dev/null || true

## Notes

- `dataset.txt` (~5.6 MB, ~100k pairs) and the BPE files ship in the repo — no rebuild needed. To regenerate: `!python build_dataset.py`.
- Progress prints every 200 iters (`train/val loss | lr`); best checkpoint saves automatically; early stopping (`patience=12`) ends the run when val loss stalls.
- Same notebook, other targets: `!python main.py --target=esp32 --train` (~10 M params, roughly 2–3× faster) or `!python main.py --target=esp8266 --train` (~1.8 M params, fastest — comfortably fits in one Kaggle session).
- Next step locally after downloading `model_esp32s3.pt.quantized`: `python flash.py esp32s3` (re-exports `src/model_weights.hpp` and flashes the S3 N16R8 board).
- Kaggle quota monitor: profile avatar > Settings > Quotas. Stop the session (Active Events > Stop) when done — idle sessions burn quota for up to 60 min.